# Stack with Ridge meta-learner (log-return targets)
Same as 98b but **targets = log returns**. Each base (LGB, TCN, Ridge, EWM) predicts **log returns**; the Ridge meta-learner also predicts **log returns** per horizon. TCN base uses best params from 07-tcn-pool; **Ridge base = ridge_core** (return lags only, best_ridge_core_params.json). Only the **final step** converts to price: `prices = p0 * np.exp(np.cumsum(pred_log_returns))`. Evaluation: RMSE, MAE, MAPE on **price errors** (dollars and %).

In [1]:
import sys
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge
import lightgbm as lgb

REPO_ROOT = Path.cwd().parent.parent
BACKEND_DIR = REPO_ROOT / "backend"
sys.path.insert(0, str(BACKEND_DIR))
sys.path.insert(0, str(Path.cwd()))

from _pool_common import (
    load_pool_data,
    build_pooled_train_stack,
    compute_metrics_averaged_over_windows,
    metrics_to_parquet,
    fetch_cnn_fear_greed_index,
    TEST_SIZE,
    FORECAST_HORIZON,
    ROLLING_STEP,
    MIN_TRAIN_STACK,
    ARTIFACTS_DIR,
    TICKERS,
)

import json
# Load best params from 05/06 random search if available (run 05 and 06 first to generate artifacts)
_lgb_path = ARTIFACTS_DIR / "best_lgb_params.json"
if _lgb_path.exists():
    with open(_lgb_path) as f:
        LGB_PARAMS = json.load(f)
    LGB_PARAMS.setdefault("random_state", 42)
    LGB_PARAMS.setdefault("verbosity", -1)
else:
    LGB_PARAMS = dict(n_estimators=100, max_depth=4, learning_rate=0.01, random_state=42, verbosity=-1)
_ridge_path = ARTIFACTS_DIR / "best_ridge_core_params.json"
if _ridge_path.exists():
    with open(_ridge_path) as f:
        _ridge = json.load(f)
    RIDGE_ALPHA = _ridge.get("alpha", 1.0)
else:
    RIDGE_ALPHA = 1.0
EWM_SPAN = 7
ROLL_VOL_WINDOW = 7  # rolling std of returns for meta rolling_vol feature

import ast
# TCN base: load best params from 07-tcn-pool random search
_tcn_path = ARTIFACTS_DIR / "best_tcn_params.parquet"
if _tcn_path.exists():
    try:
        _tcn_df = pd.read_parquet(_tcn_path)
        if not _tcn_df.empty:
            r = _tcn_df.iloc[0]
            TCN_PARAMS = {"seq_len": int(r["seq_len"]), "epochs": int(r["epochs"]), "filters": int(r["filters"]), "kernel_size": int(r["kernel_size"]), "dilations": ast.literal_eval(r["dilations_str"]), "batch_size": 256}
            SEQ_LEN = TCN_PARAMS["seq_len"]
        else:
            TCN_PARAMS = {"seq_len": 30, "epochs": 40, "filters": 64, "kernel_size": 3, "dilations": [1, 2, 4, 8], "batch_size": 256}
            SEQ_LEN = 30
    except Exception:
        TCN_PARAMS = {"seq_len": 30, "epochs": 40, "filters": 64, "kernel_size": 3, "dilations": [1, 2, 4, 8], "batch_size": 256}
        SEQ_LEN = 30
else:
    TCN_PARAMS = {"seq_len": 30, "epochs": 40, "filters": 64, "kernel_size": 3, "dilations": [1, 2, 4, 8], "batch_size": 256}
    SEQ_LEN = 30

# Feature config: TCN vs LightGBM use different feature sets (LGB: vix_sma_5, fear_greed_change)
LAG_RETURNS = 5
RSI_PERIOD = 7
MACD_FAST, MACD_SLOW, MACD_SIGNAL = 12, 26, 9
TCN_EPOCHS_OOF = TCN_PARAMS["epochs"]
TCN_EPOCHS_FINAL = TCN_PARAMS["epochs"]
RIDGE_META_ALPHA = 0.01  # Ridge meta-learner (positive=True); meta features scaled before fit
N_FOLDS = 5  # OOF: TimeSeriesSplit so base models only see past data (no chronological leakage)

In [2]:
def _rsi(series: pd.Series, period: int) -> pd.Series:
    """RSI = 100 - 100/(1 + RS), RS = avg gain / avg loss (Wilder)."""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False).mean()
    rs = avg_gain / np.where(avg_loss != 0, avg_loss, 1e-10)
    return 100 - (100 / (1 + rs))


def build_feature_df(grp: pd.DataFrame):
    """Build features: TCN = log-return lags, volume lag, RSI, MACD; LGB = VIX 5-day SMA, cyclical month, fear_greed_change. Target = next 7 **log returns**."""
    df = grp.sort_values("timestamp").copy()
    df["close"] = df["close"].astype(float)
    df["return"] = df["close"].pct_change()
    df["log_return"] = np.log(df["close"] / df["close"].shift(1))
    for i in range(1, LAG_RETURNS + 1):
        df[f"ret_lag_{i}"] = df["log_return"].shift(i)
    if "volume" in df.columns:
        df["volume_lag_1"] = df["volume"].astype(float).shift(1)
    else:
        df["volume_lag_1"] = np.nan
    df["rsi"] = _rsi(df["close"], RSI_PERIOD)
    ema_fast = df["close"].ewm(span=MACD_FAST, adjust=False).mean()
    ema_slow = df["close"].ewm(span=MACD_SLOW, adjust=False).mean()
    df["macd_line"] = ema_fast - ema_slow
    df["macd_signal"] = df["macd_line"].ewm(span=MACD_SIGNAL, adjust=False).mean()
    if "vix" in df.columns:
        vix = df["vix"].astype(np.float64)
        df["vix_sma_5"] = vix.shift(1).rolling(5).mean()
        df["vix_velocity"] = vix.diff(1)
        df["vix_momentum"] = vix - df["vix_sma_5"]
    else:
        df["vix_sma_5"] = np.nan
        df["vix_velocity"] = np.nan
        df["vix_momentum"] = np.nan
    df["month"] = pd.to_datetime(df["timestamp"]).dt.month
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    if "fear_greed" not in df.columns:
        df["fear_greed"] = 50.0
    else:
        df["fear_greed"] = df["fear_greed"].fillna(50.0)
    df["fear_greed_lag_1"] = df["fear_greed"].shift(1)
    df["fear_greed_lag_5"] = df["fear_greed"].shift(5)
    df["fear_greed_change"] = df["fear_greed_lag_1"] - df["fear_greed_lag_5"]
    df["rolling_vol"] = df["log_return"].rolling(ROLL_VOL_WINDOW, min_periods=1).std().fillna(0).astype(np.float32)
    for h in range(1, FORECAST_HORIZON + 1):
        df[f"target_{h}"] = df["log_return"].shift(-h)
    # TCN: log-return lags, volume lag, RSI, MACD
    feature_cols_tcn = [f"ret_lag_{i}" for i in range(1, LAG_RETURNS + 1)] + [
        "volume_lag_1", "rsi", "macd_line", "macd_signal"
    ]
    # LightGBM/Ridge base: VIX 5-day SMA, vix_velocity, vix_momentum, cyclical month, fear_greed change
    feature_cols_lgb = ["vix_sma_5", "vix_velocity", "vix_momentum", "month_sin", "month_cos", "fear_greed_change"]
    target_cols = [f"target_{h}" for h in range(1, FORECAST_HORIZON + 1)]
    base_cols = ["timestamp", "close", "return", "log_return", "rolling_vol"] + feature_cols_tcn + feature_cols_lgb + target_cols
    out = df[[c for c in base_cols if c in df.columns]].copy()
    return out.dropna(), feature_cols_tcn, feature_cols_lgb, target_cols

def build_sequences(X: np.ndarray, y: np.ndarray, seq_len: int):
    """Build (n_seq, seq_len, n_feat) and (n_seq, horizon) for TCN."""
    n = len(X)
    if n < seq_len + 1:
        return None, None
    X_seq, y_seq = [], []
    for i in range(seq_len, n):
        X_seq.append(X[i - seq_len : i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)

In [3]:
try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Conv1D, Lambda
    HAS_TCN = True
except Exception:
    HAS_TCN = False

def build_tcn(seq_len: int, n_feat: int, horizon: int, filters=64, kernel_size=3, dilations=None):
    if dilations is None:
        dilations = [1, 2, 4, 8]
    model = Sequential()
    for i, d in enumerate(dilations):
        model.add(Conv1D(filters, kernel_size, dilation_rate=d, padding="causal", activation="relu",
                        input_shape=(seq_len, n_feat) if i == 0 else None))
    model.add(Conv1D(horizon, 1))
    model.add(Lambda(lambda x: x[:, -1, :]))
    model.compile(optimizer="adam", loss="mse")
    return model

def train_tcn(X_seq: np.ndarray, y_seq: np.ndarray, horizon: int, epochs=40, filters=64, kernel_size=3, dilations=None, batch_size=256):
    if not HAS_TCN or X_seq is None or len(X_seq) < 10:
        return None
    if dilations is None:
        dilations = [1, 2, 4, 8]
    seq_len, n_feat = X_seq.shape[1], X_seq.shape[2]
    model = build_tcn(seq_len, n_feat, horizon, filters=filters, kernel_size=kernel_size, dilations=dilations)
    model.fit(X_seq, y_seq, epochs=epochs, batch_size=batch_size, verbose=0)
    return model

In [4]:
def train_global_stack_ridge(stacked: pd.DataFrame, horizon: int):
    """OOF: Train base models per TimeSeriesSplit fold; stack OOF predictions to train Ridge meta-learner (scaled); then retrain base models on full training set. Returns dict for predict_stack_ridge_global."""
    pooled = build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)
    if pooled.empty:
        return None
    feat_dfs = []
    for sym in pooled["symbol"].unique():
        grp = pooled[pooled["symbol"] == sym].copy()
        try:
            feat_df, feature_cols_tcn, feature_cols_lgb, target_cols = build_feature_df(grp)
        except Exception:
            continue
        if len(feat_df) < MIN_TRAIN_STACK + horizon:
            continue
        feat_df["_symbol_"] = sym
        feat_dfs.append(feat_df)
    if not feat_dfs:
        return None
    pooled_feat = pd.concat(feat_dfs, ignore_index=True)
    pooled_feat = pooled_feat.sort_values("timestamp").reset_index(drop=True)
    pooled_feat["_sym_idx"] = pooled_feat.groupby("_symbol_").cumcount()
    n_per_sym = pooled_feat.groupby("_symbol_").size()
    n_total = len(pooled_feat)
    if n_total < 100:
        return None
    sym_to_global = {sym: np.where(pooled_feat["_symbol_"] == sym)[0] for sym in pooled_feat["_symbol_"].unique()}
    tscv = TimeSeriesSplit(n_splits=N_FOLDS)
    oof_lgb = np.full((n_total, horizon), np.nan, dtype=np.float32)
    oof_tcn = np.full((n_total, horizon), np.nan, dtype=np.float32)
    oof_ridge = np.full((n_total, horizon), np.nan, dtype=np.float32)
    oof_ewm = np.full((n_total, horizon), np.nan, dtype=np.float32)
    feature_cols_ridge = [f"ret_lag_{i}" for i in range(1, LAG_RETURNS + 1)]  # ridge_core: return lags only
    for train_idx, val_idx in tscv.split(pooled_feat):
        base_train_feat = pooled_feat.iloc[train_idx].copy()
        y_base = base_train_feat[target_cols].values.astype(np.float32)
        scaler_lgb_f = StandardScaler().fit(base_train_feat[feature_cols_lgb].values.astype(np.float32))
        scaler_tcn_f = StandardScaler().fit(base_train_feat[feature_cols_tcn].values.astype(np.float32))
        scaler_ridge_f = StandardScaler().fit(base_train_feat[feature_cols_ridge].values.astype(np.float64))
        X_lgb_base_s = scaler_lgb_f.transform(base_train_feat[feature_cols_lgb].values.astype(np.float32))
        X_tcn_base_s = scaler_tcn_f.transform(base_train_feat[feature_cols_tcn].values.astype(np.float32))
        lgb_multi = MultiOutputRegressor(lgb.LGBMRegressor(**LGB_PARAMS))
        lgb_multi.fit(X_lgb_base_s, y_base)
        X_seq_list, y_seq_list = [], []
        for sym in base_train_feat["_symbol_"].unique():
            sym_mask = base_train_feat["_symbol_"] == sym
            X_s = X_tcn_base_s[sym_mask]
            y_s = y_base[sym_mask]
            X_seq, y_seq = build_sequences(X_s, y_s, SEQ_LEN)
            if X_seq is not None and len(X_seq) >= 10:
                X_seq_list.append(X_seq)
                y_seq_list.append(y_seq)
        tcn_model = train_tcn(np.concatenate(X_seq_list, axis=0), np.concatenate(y_seq_list, axis=0), horizon, epochs=TCN_EPOCHS_OOF, filters=TCN_PARAMS["filters"], kernel_size=TCN_PARAMS["kernel_size"], dilations=TCN_PARAMS["dilations"], batch_size=TCN_PARAMS["batch_size"]) if X_seq_list else None
        X_ridge_base_s = scaler_ridge_f.transform(base_train_feat[feature_cols_ridge].values.astype(np.float64))
        ridge_multi = MultiOutputRegressor(Ridge(alpha=RIDGE_ALPHA, random_state=42))
        ridge_multi.fit(X_ridge_base_s, y_base.astype(np.float64))
        X_lgb_full_s = scaler_lgb_f.transform(pooled_feat[feature_cols_lgb].values.astype(np.float32))
        X_tcn_full_s = scaler_tcn_f.transform(pooled_feat[feature_cols_tcn].values.astype(np.float32))
        X_ridge_full_s = scaler_ridge_f.transform(pooled_feat[feature_cols_ridge].values.astype(np.float64))
        valid_val_list = []
        for i in val_idx:
            sym = pooled_feat.iloc[i]["_symbol_"]
            sym_idx = int(pooled_feat.iloc[i]["_sym_idx"])
            n_sym = int(n_per_sym[sym])
            if sym_idx < SEQ_LEN or sym_idx >= n_sym - horizon:
                continue
            valid_val_list.append(i)
        if valid_val_list:
            valid_idx = np.array(valid_val_list)
            oof_lgb[valid_idx] = lgb_multi.predict(X_lgb_full_s[valid_idx])
            seq_list = []
            for i in valid_idx:
                sym = pooled_feat.iloc[i]["_symbol_"]
                sym_idx = int(pooled_feat.iloc[i]["_sym_idx"])
                gidx = sym_to_global[sym]
                seq_idx = gidx[sym_idx - SEQ_LEN : sym_idx]
                seq_list.append(X_tcn_full_s[seq_idx])
            seq_batch = np.array(seq_list)
            if tcn_model is not None:
                oof_tcn[valid_idx] = tcn_model.predict(seq_batch, verbose=0)
            else:
                oof_tcn[valid_idx] = oof_lgb[valid_idx]
            oof_ridge[valid_idx] = ridge_multi.predict(X_ridge_full_s[valid_idx])
            for i in valid_idx:
                sym = pooled_feat.iloc[i]["_symbol_"]
                sym_idx = int(pooled_feat.iloc[i]["_sym_idx"])
                mask = (pooled_feat["_symbol_"] == sym) & (pooled_feat["_sym_idx"] <= sym_idx)
                r = pooled_feat.loc[mask, "log_return"].values.astype(np.float64)
                if len(r) >= EWM_SPAN:
                    ewm_val = float(pd.Series(r).ewm(span=EWM_SPAN).mean().iloc[-1])
                    oof_ewm[i, :] = ewm_val
    y = pooled_feat[target_cols].values.astype(np.float32)
    valid = ~(np.any(np.isnan(oof_lgb), axis=1) | np.any(np.isnan(oof_tcn), axis=1) | np.any(np.isnan(oof_ridge), axis=1) | np.any(np.isnan(oof_ewm), axis=1))
    if np.sum(valid) < 5:
        linear_models = []
        meta_scaler = None
    else:
        meta_X_lgb = oof_lgb[valid]
        meta_X_tcn = oof_tcn[valid]
        meta_X_ridge = oof_ridge[valid]
        meta_X_ewm = oof_ewm[valid]
        # OOF base-prediction correlation (per horizon, then mean); high Ridge–LGB/TCN => Ridge may be redundant
        names = ["lgb", "tcn", "ridge", "ewm"]
        corr_sum = np.zeros((4, 4))
        n_h = 0
        for h in range(horizon):
            arr = np.column_stack([meta_X_lgb[:, h], meta_X_tcn[:, h], meta_X_ridge[:, h], meta_X_ewm[:, h]])
            if np.any(~np.isfinite(arr)):
                continue
            c = np.corrcoef(arr.T)
            if c is not None and c.shape == (4, 4):
                corr_sum += c
                n_h += 1
        if n_h > 0:
            corr_mean = corr_sum / n_h
            oof_corr_df = pd.DataFrame(corr_mean, index=names, columns=names)
            print("OOF base-prediction correlation (mean over horizons):")
            print(oof_corr_df.round(4).to_string())
            r_lgb_ridge = corr_mean[0, 2]
            r_tcn_ridge = corr_mean[1, 2]
            if r_lgb_ridge > 0.9 or r_tcn_ridge > 0.9:
                print("  -> Ridge vs LGB corr = {:.4f}, Ridge vs TCN corr = {:.4f}. High correlation supports Ridge being redundant (meta may zero it out).".format(r_lgb_ridge, r_tcn_ridge))
        valid_df = pooled_feat.iloc[np.where(valid)[0]]
        meta_ctx = valid_df[["month_sin", "month_cos", "vix_velocity", "rolling_vol"]].values.astype(np.float32)
        meta_y = y[valid]
        X_meta_list = []
        for h in range(horizon):
            disagreement_h = np.std(np.column_stack([meta_X_lgb[:, h], meta_X_tcn[:, h], meta_X_ridge[:, h], meta_X_ewm[:, h]]), axis=1, keepdims=True).astype(np.float32)
            X_meta_list.append(np.hstack([meta_X_lgb[:, h : h + 1], meta_X_tcn[:, h : h + 1], meta_X_ridge[:, h : h + 1], meta_X_ewm[:, h : h + 1], disagreement_h, meta_ctx]))
        X_meta_all = np.vstack(X_meta_list)
        meta_scaler = StandardScaler()
        meta_scaler.fit(X_meta_all)
        linear_models = []
        for h in range(horizon):
            disagreement_h = np.std(np.column_stack([meta_X_lgb[:, h], meta_X_tcn[:, h], meta_X_ridge[:, h], meta_X_ewm[:, h]]), axis=1, keepdims=True).astype(np.float32)
            X_meta = np.hstack([meta_X_lgb[:, h : h + 1], meta_X_tcn[:, h : h + 1], meta_X_ridge[:, h : h + 1], meta_X_ewm[:, h : h + 1], disagreement_h, meta_ctx])
            X_meta_s = meta_scaler.transform(X_meta)
            ridge_meta = Ridge(alpha=RIDGE_META_ALPHA, positive=True, random_state=42)
            ridge_meta.fit(X_meta_s, meta_y[:, h])
            linear_models.append(ridge_meta)
    # Final base training on full training set (one scaler per feature set)
    y_full = pooled_feat[target_cols].values.astype(np.float32)
    scaler_lgb = StandardScaler().fit(pooled_feat[feature_cols_lgb].values.astype(np.float32))
    scaler_tcn = StandardScaler().fit(pooled_feat[feature_cols_tcn].values.astype(np.float32))
    scaler_ridge = StandardScaler().fit(pooled_feat[feature_cols_ridge].values.astype(np.float64))
    X_lgb_full_s = scaler_lgb.transform(pooled_feat[feature_cols_lgb].values.astype(np.float32))
    X_tcn_full_s = scaler_tcn.transform(pooled_feat[feature_cols_tcn].values.astype(np.float32))
    X_ridge_full_s = scaler_ridge.transform(pooled_feat[feature_cols_ridge].values.astype(np.float64))
    lgb_multi = MultiOutputRegressor(lgb.LGBMRegressor(**LGB_PARAMS))
    lgb_multi.fit(X_lgb_full_s, y_full)
    X_seq_list, y_seq_list = [], []
    for sym in pooled_feat["_symbol_"].unique():
        sym_mask = pooled_feat["_symbol_"] == sym
        X_s = X_tcn_full_s[sym_mask]
        y_s = y_full[sym_mask]
        X_seq, y_seq = build_sequences(X_s, y_s, SEQ_LEN)
        if X_seq is not None and len(X_seq) >= 10:
            X_seq_list.append(X_seq)
            y_seq_list.append(y_seq)
    tcn_model = train_tcn(np.concatenate(X_seq_list, axis=0), np.concatenate(y_seq_list, axis=0), horizon, epochs=TCN_EPOCHS_FINAL, filters=TCN_PARAMS["filters"], kernel_size=TCN_PARAMS["kernel_size"], dilations=TCN_PARAMS["dilations"], batch_size=TCN_PARAMS["batch_size"]) if X_seq_list else None
    ridge_multi = MultiOutputRegressor(Ridge(alpha=RIDGE_ALPHA, random_state=42))
    ridge_multi.fit(X_ridge_full_s, y_full.astype(np.float64))
    return {
        "scaler_lgb": scaler_lgb, "scaler_tcn": scaler_tcn, "scaler_ridge": scaler_ridge,
        "meta_scaler": meta_scaler,
        "lgb_multi": lgb_multi, "tcn_model": tcn_model, "ridge_multi": ridge_multi, "linear_models": linear_models,
        "feature_cols_tcn": feature_cols_tcn, "feature_cols_lgb": feature_cols_lgb, "feature_cols_ridge": feature_cols_ridge,
        "target_cols": target_cols, "EWM_SPAN": EWM_SPAN, "seq_len": SEQ_LEN,
    }


def predict_stack_ridge_global(context_df: pd.DataFrame, horizon: int, global_stack: dict) -> list:
    """Bases + meta all output log returns; only here convert to price: prices = p0 * exp(cumsum(pred_log_returns))."""
    if global_stack is None or not global_stack.get("linear_models") or global_stack.get("meta_scaler") is None:
        return []
    try:
        feat_df, feature_cols_tcn, feature_cols_lgb, _ = build_feature_df(context_df)
    except Exception:
        return []
    seq_len = int(global_stack.get("seq_len", 30))
    if len(feat_df) < seq_len + 1:
        return []
    feature_cols_ridge = global_stack.get("feature_cols_ridge", [f"ret_lag_{i}" for i in range(1, LAG_RETURNS + 1)])
    scaler_lgb = global_stack.get("scaler_lgb")
    scaler_tcn = global_stack.get("scaler_tcn")
    scaler_ridge = global_stack.get("scaler_ridge")
    if scaler_lgb is None or scaler_tcn is None or scaler_ridge is None:
        return []
    lgb_multi = global_stack["lgb_multi"]
    tcn_model = global_stack["tcn_model"]
    ridge_multi = global_stack["ridge_multi"]
    linear_models = global_stack["linear_models"]
    EWM_SPAN = global_stack.get("EWM_SPAN", 7)
    X_lgb_s = scaler_lgb.transform(feat_df[feature_cols_lgb].values.astype(np.float32))
    X_tcn_s = scaler_tcn.transform(feat_df[feature_cols_tcn].values.astype(np.float32))
    X_ridge_s = scaler_ridge.transform(feat_df[feature_cols_ridge].values.astype(np.float64))
    last_idx = len(feat_df) - 1
    last_row_lgb = X_lgb_s[last_idx : last_idx + 1]
    last_row_ridge = X_ridge_s[last_idx : last_idx + 1]
    lgb_21 = lgb_multi.predict(last_row_lgb).ravel()
    ridge_21 = ridge_multi.predict(last_row_ridge).ravel()
    log_ret_series = feat_df["log_return"].values.astype(np.float64)
    if len(log_ret_series) >= EWM_SPAN:
        ewm_val = float(pd.Series(log_ret_series).ewm(span=EWM_SPAN).mean().iloc[-1])
        ewm_21 = np.full(horizon, ewm_val, dtype=np.float32)
    else:
        ewm_21 = np.full(horizon, float(np.nanmean(log_ret_series)) if len(log_ret_series) else 0.0, dtype=np.float32)
    if tcn_model is not None and last_idx >= seq_len:
        seq = X_tcn_s[last_idx - seq_len : last_idx]
        tcn_21 = tcn_model.predict(seq.reshape(1, seq_len, -1), verbose=0).ravel()
    else:
        tcn_21 = lgb_21
    vv = feat_df["vix_velocity"].iloc[-1]
    vix_vel = np.nan_to_num(float(vv), nan=0.0)
    roll_vol = float(feat_df["rolling_vol"].iloc[-1]) if "rolling_vol" in feat_df.columns else 0.0
    ctx_last = np.concatenate([feat_df[["month_sin", "month_cos"]].iloc[-1].values.astype(np.float32), [np.float32(vix_vel), np.float32(roll_vol)]])
    meta_vec = np.array([np.concatenate([np.array([lgb_21[h], tcn_21[h], ridge_21[h], ewm_21[h], np.std([lgb_21[h], tcn_21[h], ridge_21[h], ewm_21[h]])], dtype=np.float32), ctx_last]) for h in range(horizon)])
    meta_vec_s = global_stack["meta_scaler"].transform(meta_vec)
    # Meta predicts log return per step; only final step: log returns -> price
    final_log_returns = np.array([linear_models[h].predict(meta_vec_s[h : h + 1])[0] for h in range(horizon)])
    p0 = float(context_df["close"].iloc[-1])
    prices = p0 * np.exp(np.cumsum(final_log_returns))
    return [float(p) for p in prices[:horizon]]


In [5]:
stacked = load_pool_data(with_vix=True, with_volume=True)
symbol_start = pd.to_datetime(stacked["timestamp"]).min().strftime("%Y-%m-%d")
fear_greed_df = fetch_cnn_fear_greed_index(start_date=symbol_start)
if not fear_greed_df.empty:
    stacked["date"] = pd.to_datetime(stacked["timestamp"]).dt.normalize()
    fear_greed_df["date"] = pd.to_datetime(fear_greed_df["timestamp"]).dt.normalize()
    stacked = stacked.merge(fear_greed_df[["date", "fear_greed"]], on="date", how="left")
    stacked["fear_greed"] = stacked["fear_greed"].ffill().bfill()
    stacked = stacked.drop(columns=["date"])
print(stacked.groupby("symbol").size())
stacked.head(10)

symbol
AAPL     1256
AMZN     1256
GOOGL    1256
JNJ      1256
JPM      1256
MSFT     1256
NVDA     1256
SPY      1256
WMT      1256
XOM      1256
dtype: int64


,timestamp,symbol,close,volume,vix,fear_greed
0,2021-03-01,AAPL,127.790001,116307900,23.350000,58.52
1,2021-03-02,AAPL,125.120003,102260900,24.100000,50.32
2,2021-03-03,AAPL,122.059998,112966300,26.670000,43.84
3,2021-03-04,AAPL,120.129997,178155000,28.570000,37.80
4,2021-03-05,AAPL,121.419998,153766600,24.660000,41.52
5,2021-03-08,AAPL,116.360001,154376600,25.469999,39.08
6,2021-03-09,AAPL,121.089996,129525800,24.030001,43.36
7,2021-03-10,AAPL,119.980003,111943300,22.559999,45.56
8,2021-03-11,AAPL,121.959999,103026500,21.910000,50.48
9,2021-03-12,AAPL,121.029999,88105100,20.690001,53.72


In [6]:
stacked.describe()

,timestamp,close,volume,vix,fear_greed
count,12560,12560.000000,1.256000e+04,12560.000000,12560.000000
mean,2023-08-27 08:25:36.305000,194.495601,7.095065e+07,19.147627,47.554319
min,2021-03-01 00:00:00,11.227000,2.316700e+06,11.860000,2.900000
25%,2022-05-25 18:00:00,109.129997,1.490855e+07,15.487500,33.150000
50%,2023-08-26 12:00:00,158.250000,2.869340e+07,17.889999,48.021429
75%,2024-11-22 18:00:00,234.557499,6.313702e+07,21.612500,62.948810
max,2026-02-27 00:00:00,695.489990,1.543911e+09,52.330002,82.971429
std,NaN,138.142456,1.247986e+08,5.201849,18.318310


In [7]:
# Train once on pooled data (all assets, only rows before 60-day test window)
global_stack = train_global_stack_ridge(stacked, FORECAST_HORIZON)
pooled_train = build_pooled_train_stack(stacked, TEST_SIZE, MIN_TRAIN_STACK)
print("Global stack trained on", len(pooled_train), "pooled train rows.")

c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init

OOF base-prediction correlation (mean over horizons):
          lgb     tcn   ridge     ewm
lgb    1.0000  0.0403  0.1313 -0.1056
tcn    0.0403  1.0000  0.0254 -0.0078
ridge  0.1313  0.0254  1.0000  0.0175
ewm   -0.1056 -0.0078  0.0175  1.0000


c:\capstone_project_unfc\env\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Global stack trained on 11960 pooled train rows.


In [8]:
# Backtest: 60-day test window, 7-day horizon, step 7 days; predict only (global models trained once above)
model_name = "stack_ridge_meta_logreturn"
all_preds = []
for sym in TICKERS:
    grp = stacked[stacked["symbol"] == sym].copy()
    if grp.empty:
        continue
    grp = grp.sort_values("timestamp").reset_index(drop=True)
    prices = grp.set_index("timestamp")["close"].astype(float).dropna()
    n = len(prices)
    if n < TEST_SIZE + MIN_TRAIN_STACK:
        continue
    split_idx = n - TEST_SIZE
    test_index = prices.index[split_idx:]
    test_values = prices.values[split_idx:]
    preds = []
    window_ix = 0
    start = 0
    while start + FORECAST_HORIZON <= TEST_SIZE:
        context_cols = ["timestamp", "close", "vix", "symbol"] + [c for c in ["volume", "fear_greed"] if c in grp.columns]
        context_df = grp.iloc[: split_idx + start][context_cols].copy()
        if len(context_df) < MIN_TRAIN_STACK:
            start += ROLLING_STEP
            continue
        price_list = predict_stack_ridge_global(context_df, FORECAST_HORIZON, global_stack)
        if not price_list or len(price_list) < FORECAST_HORIZON:
            start += ROLLING_STEP
            window_ix += 1
            continue
        for h in range(FORECAST_HORIZON):
            idx = start + h
            ts = test_index[idx]
            y_true = float(test_values[idx])
            y_pred = float(price_list[h])
            preds.append({"timestamp": ts, "y_true": y_true, "y_pred": y_pred, "window_ix": window_ix})
        window_ix += 1
        start += ROLLING_STEP
    if preds:
        pred_df = pd.DataFrame(preds)
        pred_df["symbol"] = sym
        all_preds.append(pred_df)

pred_stack = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame(
    columns=["timestamp", "y_true", "y_pred", "window_ix", "symbol"]
)
print(pred_stack.groupby("symbol").size() if not pred_stack.empty else "No predictions.")
pred_stack.head()


symbol
AAPL     56
AMZN     56
GOOGL    56
JNJ      56
JPM      56
MSFT     56
NVDA     56
SPY      56
WMT      56
XOM      56
dtype: int64


,timestamp,y_true,y_pred,window_ix,symbol
0,2025-12-02,286.190002,283.333893,0,AAPL
1,2025-12-03,284.149994,283.545929,0,AAPL
2,2025-12-04,280.700012,283.544983,0,AAPL
3,2025-12-05,278.779999,283.619324,0,AAPL
4,2025-12-08,277.890015,283.779938,0,AAPL


In [9]:
# Global stack is trained in the cell above. Run that cell before the backtest.

In [10]:
metrics_rows = []
for sym in pred_stack["symbol"].unique():
    sub = pred_stack[pred_stack["symbol"] == sym]
    m = compute_metrics_averaged_over_windows(sub)
    metrics_rows.append({"model": model_name, "symbol": sym, **m})
m_overall = compute_metrics_averaged_over_windows(pred_stack)
metrics_rows.append({"model": model_name, "symbol": "overall", **m_overall})

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string())
metrics_to_parquet(metrics_rows, ARTIFACTS_DIR / "metrics_stack_ridge_meta_logreturn_pool.parquet")
print("Saved:", ARTIFACTS_DIR / "metrics_stack_ridge_meta_logreturn_pool.parquet")

                         model   symbol        MAE       RMSE    MAPE_%
0   stack_ridge_meta_logreturn     AAPL   6.759644   7.609835  2.565839
1   stack_ridge_meta_logreturn     MSFT  11.022739  12.024642  2.500130
2   stack_ridge_meta_logreturn    GOOGL   8.149086   9.200439  2.568472
3   stack_ridge_meta_logreturn     AMZN   8.955647   9.914837  3.990209
4   stack_ridge_meta_logreturn      JPM   7.385436   8.178844  2.355845
5   stack_ridge_meta_logreturn      JNJ   4.043655   4.469285  1.812084
6   stack_ridge_meta_logreturn      WMT   2.512159   2.841140  2.089898
7   stack_ridge_meta_logreturn      SPY   6.704443   7.596637  0.978319
8   stack_ridge_meta_logreturn      XOM   4.524202   4.877608  3.309047
9   stack_ridge_meta_logreturn     NVDA   4.270799   4.946392  2.341829
10  stack_ridge_meta_logreturn  overall   6.432781   8.406946  2.451167
Saved: C:\capstone_project_unfc\model\experiments-pool\artifacts\metrics_stack_ridge_meta_logreturn_pool.parquet


In [11]:
# Ridge meta-learner coefficients (one per horizon; features: lgb, tcn, ridge, ewm, disagreement, month_sin, month_cos, vix_velocity, rolling_vol, symbol one-hot)
sym = TICKERS[0]
grp = stacked[stacked["symbol"] == sym].sort_values("timestamp").reset_index(drop=True)
n = len(grp)
if n >= TEST_SIZE + MIN_TRAIN_STACK:
    linear_models = (global_stack or {}).get("linear_models", [])
    meta_feature_names = ["coef_lgb", "coef_tcn", "coef_ridge", "coef_ewm", "coef_disagreement", "coef_month_sin", "coef_month_cos", "coef_vix_velocity", "coef_rolling_vol"]
    if linear_models:
        rows = []
        for h, ridge_meta in enumerate(linear_models):
            coef = ridge_meta.coef_
            rows.append({"step": h + 1, **dict(zip(meta_feature_names, coef))})
        meta_df = pd.DataFrame(rows)
        print("Ridge meta-learner coefficients (lgb, tcn, ridge, ewm, disagreement, month_sin, month_cos, vix_velocity, rolling_vol, symbol one-hot)")
        print(meta_df.to_string(index=False))
    else:
        print("Could not obtain linear models.")
else:
    print("Not enough data for meta-learner coefficients.")

Ridge meta-learner coefficients (lgb, tcn, ridge, ewm, disagreement, month_sin, month_cos, vix_velocity, rolling_vol, symbol one-hot)
 step  coef_lgb  coef_tcn  coef_ridge  coef_ewm  coef_disagreement  coef_month_sin  coef_month_cos  coef_vix_velocity  coef_rolling_vol
    1  0.000708  0.000185         0.0  0.000000           0.000262             0.0        0.000039           0.000344          0.000000
    2  0.000000  0.000000         0.0  0.000000           0.000000             0.0        0.000105           0.000000          0.000182
    3  0.000000  0.000000         0.0  0.000000           0.000384             0.0        0.000030           0.000794          0.000108
    4  0.000000  0.000000         0.0  0.000007           0.000000             0.0        0.000000           0.000402          0.000379
    5  0.000091  0.000049         0.0  0.000000           0.000000             0.0        0.000000           0.000000          0.000262
    6  0.000000  0.000488         0.0  0.000010   